In [10]:
def to_twos_complement_hex(val, bits):
    """Converts a signed integer to a zero-padded hex string."""
    mask = (1 << bits) - 1
    hex_chars = (bits + 3) // 4  
    return f"0x{(val & mask):0{hex_chars}X}"

def from_twos_complement_hex(hex_val, bits):
    """Converts a hardware hex literal into a signed Python integer."""
    if hex_val & (1 << (bits - 1)):
        hex_val -= (1 << bits)
    return hex_val

def hardware_add_extended(a, b, input_width):
    """
    Simulates hardware addition where the output width grows by 1 bit 
    to completely prevent wrap-around overflow.
    """
    output_width = input_width + 1
    
    # 1. Standard addition
    raw_sum = a + b
    
    # 2. Simulate storing the sum in an extended 17-bit register
    mask_n = (1 << output_width) - 1
    sum_hw = raw_sum & mask_n
    
    # 3. Convert the raw hardware bits back to a signed Python integer
    if sum_hw & (1 << (output_width - 1)):
        return sum_hw - (1 << output_width)
    return sum_hw

def run_adder_simulation():
    # Scale factor for 15 fractional bits (2^15)
    SCALE = 32768.0
    INPUT_BITS = 16
    OUTPUT_BITS = 17 # Q2.15 format

    # Hex inputs extracted exactly from your Verilog testbench
    test_cases_hex = [
        (0x0180, 0x0280),   # TC1: 0.0117 + 0.0195
        (0xFE80, 0x0080),   # TC2: -0.0117 + 0.0039
        (0x7FFF, 0x7FFF),   # TC3: Max positive
        (0x8000, 0x8000),   # TC4: Max negative
        (0x03E8, 0xFC18)    # TC5: Zero Crossing
    ]

    print("Starting Python Q2.15 Extended Adder Simulation...")
    print("=========================================================================================")

    for idx, (a_hex_in, b_hex_in) in enumerate(test_cases_hex, 1):
        # Decode hex to 16-bit signed integer
        a = from_twos_complement_hex(a_hex_in, INPUT_BITS)
        b = from_twos_complement_hex(b_hex_in, INPUT_BITS)

        # Emulate the Verilog 17-bit hardware addition
        sum_val = hardware_add_extended(a, b, input_width=INPUT_BITS)

        # Convert to floating point for readability
        a_float = a / SCALE
        b_float = b / SCALE
        sum_float = sum_val / SCALE

        # Encode back to hardware hex for output verification
        # Inputs are passed as 16-bit, sum is passed as 17-bit
        a_hex_out = to_twos_complement_hex(a, INPUT_BITS)
        b_hex_out = to_twos_complement_hex(b, INPUT_BITS)
        sum_hex_out = to_twos_complement_hex(sum_val, OUTPUT_BITS)

        # Print results. Notice the sum requires an extra hex character (5 chars)
        print(f"Time={idx * 15} | a = {a_float:8.4f} ({a_hex_out}) | b = {b_float:8.4f} ({b_hex_out}) | sum = {sum_float:8.4f} ({sum_hex_out})")

if __name__ == "__main__":
    run_adder_simulation()

Starting Python Q2.15 Extended Adder Simulation...
Time=15 | a =   0.0117 (0x0180) | b =   0.0195 (0x0280) | sum =   0.0312 (0x00400)
Time=30 | a =  -0.0117 (0xFE80) | b =   0.0039 (0x0080) | sum =  -0.0078 (0x1FF00)
Time=45 | a =   1.0000 (0x7FFF) | b =   1.0000 (0x7FFF) | sum =   1.9999 (0x0FFFE)
Time=60 | a =  -1.0000 (0x8000) | b =  -1.0000 (0x8000) | sum =  -2.0000 (0x10000)
Time=75 | a =   0.0305 (0x03E8) | b =  -0.0305 (0xFC18) | sum =   0.0000 (0x00000)


In [9]:
def to_twos_complement_hex(val, bits):
    """Converts a signed integer to a zero-padded hex string."""
    mask = (1 << bits) - 1
    hex_chars = (bits + 3) // 4  
    return f"0x{(val & mask):0{hex_chars}X}"

def from_twos_complement_hex(hex_val, bits):
    """Converts a hardware hex literal into a signed Python integer."""
    if hex_val & (1 << (bits - 1)):
        hex_val -= (1 << bits)
    return hex_val

def hardware_multiply(a, b, int_width, frac_width):
    """
    Simulates the exact bit-level truncation and saturation logic of the Verilog multiplier.
    """
    total_width = int_width + frac_width
    
    # Calculate dynamic Min and Max saturation values (matching Verilog logic)
    min_val = -(1 << (total_width - 1))        # e.g., -32768 for 16-bit
    max_val = (1 << (total_width - 1)) - 1     # e.g., 32767 for 16-bit
    
    # ---------------------------------------------------------
    # Saturation Logic for the -1.0 * -1.0 Edge Case
    # ---------------------------------------------------------
    # If both inputs are exactly the minimum negative value, clamp the output 
    # to the maximum positive value.
    if a == min_val and b == min_val:
        return max_val
        
    # 1. Full precision hardware multiplication
    full_product = a * b
    
    # 2. Simulate a hardware register of 2*N bits (32 bits for Q1.15)
    mask_2n = (1 << (2 * total_width)) - 1
    hw_full_product = full_product & mask_2n
    
    # 3. Apply the part-select: Shift right by FRAC_WIDTH and mask to TOTAL_WIDTH
    mask_n = (1 << total_width) - 1
    prod_raw = (hw_full_product >> frac_width) & mask_n
    
    # 4. Convert the raw hardware bits back to a signed Python integer
    if prod_raw & (1 << (total_width - 1)):
        return prod_raw - (1 << total_width)
    return prod_raw

def run_multiplier_simulation():
    # Scale factor for Q1.15 (2^15)
    SCALE = 32768.0
    TOTAL_BITS = 16

    # Test cases mapped exactly to the Verilog TB
    test_cases_hex = [
        (0x4000, 0x4000),   # TC1:  0.5 * 0.5
        (0xC000, 0x2000),   # TC2: -0.5 * 0.25
        (0x1000, 0x1000),   # TC3:  0.125 * 0.125
        (0xA000, 0xA000),   # TC4: -0.75 * -0.75
        (0x8000, 0x8000)    # TC5: -1.0 * -1.0 (Now correctly saturating)
    ]

    print("Starting Python Q1.15 Multiplier Simulation (With Saturation)...")
    print("=" * 75)

    for idx, (a_hex_in, b_hex_in) in enumerate(test_cases_hex, 1):
        # Decode hex to signed integer
        a = from_twos_complement_hex(a_hex_in, TOTAL_BITS)
        b = from_twos_complement_hex(b_hex_in, TOTAL_BITS)

        # Emulate the Verilog hardware logic with Q1.15 parameters
        prod = hardware_multiply(a, b, int_width=1, frac_width=15)

        # Convert to floating point for readability
        a_float = a / SCALE
        b_float = b / SCALE
        prod_float = prod / SCALE

        # Encode back to hardware hex for output verification
        a_hex_out = to_twos_complement_hex(a, TOTAL_BITS)
        b_hex_out = to_twos_complement_hex(b, TOTAL_BITS)
        prod_hex_out = to_twos_complement_hex(prod, TOTAL_BITS)

        # Print results mimicking $monitor
        print(f"Test Case {idx}:")
        print(f"  a    = {a_float:8.4f} ({a_hex_out})")
        print(f"  b    = {b_float:8.4f} ({b_hex_out})")
        print(f"  prod = {prod_float:8.4f} ({prod_hex_out})")
        print("-" * 75)

if __name__ == "__main__":
    run_multiplier_simulation()

Starting Python Q1.15 Multiplier Simulation (With Saturation)...
Test Case 1:
  a    =   0.5000 (0x4000)
  b    =   0.5000 (0x4000)
  prod =   0.2500 (0x2000)
---------------------------------------------------------------------------
Test Case 2:
  a    =  -0.5000 (0xC000)
  b    =   0.2500 (0x2000)
  prod =  -0.1250 (0xF000)
---------------------------------------------------------------------------
Test Case 3:
  a    =   0.1250 (0x1000)
  b    =   0.1250 (0x1000)
  prod =   0.0156 (0x0200)
---------------------------------------------------------------------------
Test Case 4:
  a    =  -0.7500 (0xA000)
  b    =  -0.7500 (0xA000)
  prod =   0.5625 (0x4800)
---------------------------------------------------------------------------
Test Case 5:
  a    =  -1.0000 (0x8000)
  b    =  -1.0000 (0x8000)
  prod =   1.0000 (0x7FFF)
---------------------------------------------------------------------------
